# Stoppage Event Detection

Find clips where motion drops sharply in the last 5-10 seconds using only each clip's `_motion_events.csv`.


In [81]:
import re
import sys
from pathlib import Path

import boto3
import cv2
import numpy as np
import pandas as pd

print(sys.executable)
print(Path.cwd())

def read_preview_frame(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    ok, frame = cap.read()
    cap.release()
    return frame if ok else None


c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\.venv\Scripts\python.exe
c:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights


In [82]:
PROFILE = "DashcamGlbDiageoProdDataContrib-522196013725"
BUCKET = "diageo-prod-global-dashcam-mc-nuc-video"
PREFIX = "cortexvpu-01a-005-41884872/"
VIDEO_EXTS = (".ts", ".mp4", ".mov", ".mkv", ".avi")
MOTION_EVENTS_SUFFIX = "_motion_events.csv"

START_TIME = "2026-06-18 21:00:00"
END_TIME = "2026-06-19 01:30:00"

MAX_KEYS = 10
MAX_CANDIDATES = 10
TAIL_WINDOW_SECONDS = 5
MAX_TAIL_TRIGGER_RATIO = 0.02
MIN_TRIGGER_RATIO_DROP = 0.0
MIN_PRE_TAIL_ROWS = 1
MIN_TAIL_ROWS = 2
CV_TAIL_SECONDS = 5
CV_SAMPLE_FPS = 2
CV_RESIZE_WIDTH = 500
ROI = None

PROCESS_FLAGGED_DIR = Path("process_flagged")
DOWNLOAD_DIR = PROCESS_FLAGGED_DIR / "downloads"
PROCESS_FLAGGED_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)


In [83]:
session = boto3.Session(profile_name=PROFILE) if PROFILE else boto3.Session()
s3 = session.client("s3")
print(f"Connected to s3://{BUCKET}/{PREFIX}")
            


Connected to s3://diageo-prod-global-dashcam-mc-nuc-video/cortexvpu-01a-005-41884872/


In [84]:
timestamp_re = re.compile(r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2}_\d+)")
start_ts = pd.to_datetime(START_TIME) if START_TIME else None
end_ts = pd.to_datetime(END_TIME) if END_TIME else None

all_objects = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    all_objects.extend(page.get("Contents", []))

objects = all_objects if MAX_KEYS is None else all_objects[:MAX_KEYS]

motion_event_files = {}
for obj in objects:
    key = obj["Key"]
    if not key.endswith(MOTION_EVENTS_SUFFIX):
        continue
    stem = Path(key).name[:-len(MOTION_EVENTS_SUFFIX)]
    motion_event_files[stem] = {
        "motion_events_key": key,
        "motion_events_size_bytes": obj["Size"],
        "motion_events_size_mb": obj["Size"] / (1024 * 1024),
    }

rows = []
for obj in objects:
    key = obj["Key"]
    if not key.lower().endswith(VIDEO_EXTS):
        continue
    match = timestamp_re.search(key)
    if not match:
        continue

    clip_start = pd.to_datetime(match.group(1), format="%Y-%m-%d_%H-%M-%S_%f")
    if start_ts is not None and clip_start < start_ts:
        continue
    if end_ts is not None and clip_start > end_ts:
        continue

    stem = Path(key).stem
    row = {
        "key": key,
        "timestamp": clip_start,
        "size_bytes": obj["Size"],
        "size_mb": obj["Size"] / (1024 * 1024),
    }
    row.update(motion_event_files.get(stem, {
        "motion_events_key": None,
        "motion_events_size_bytes": None,
        "motion_events_size_mb": None,
    }))
    rows.append(row)

df = pd.DataFrame(rows, columns=[
    "key",
    "timestamp",
    "size_bytes",
    "size_mb",
    "motion_events_key",
    "motion_events_size_bytes",
    "motion_events_size_mb",
])
if not df.empty:
    df = df.sort_values("timestamp").reset_index(drop=True)
else:
    print("No videos matched the current prefix and time range.")
    print(f"START_TIME={START_TIME!r}, END_TIME={END_TIME!r}, PREFIX={PREFIX!r}")

df["next_timestamp"] = df["timestamp"].shift(-1)
df["start_to_start_seconds"] = (df["next_timestamp"] - df["timestamp"]).dt.total_seconds()

print("Total objects under prefix:", len(all_objects))
print("Motion events files found:", len(motion_event_files))
print("Video clips in selected range:", len(df))
print("Videos with motion_events csv:", int(df["motion_events_key"].notna().sum()))
df[["timestamp", "key", "motion_events_key", "start_to_start_seconds"]].head()


No videos matched the current prefix and time range.
START_TIME='2026-06-18 21:00:00', END_TIME='2026-06-19 01:30:00', PREFIX='cortexvpu-01a-005-41884872/'


AttributeError: Can only use .dt accessor with datetimelike values

In [ ]:
motion_summary_rows = []
for row in df[["key", "motion_events_key"]].drop_duplicates().itertuples(index=False):
    key = row.key
    motion_key = row.motion_events_key

    if pd.isna(motion_key) or motion_key is None:
        motion_summary_rows.append({
            "key": key,
            "motion_events_rows": 0,
            "trigger_count": None,
            "trigger_ratio": None,
            "pre_tail_rows": None,
            "pre_tail_trigger_count": None,
            "pre_tail_trigger_ratio": None,
            "tail_rows": None,
            "tail_trigger_count": None,
            "tail_trigger_ratio": None,
            "trigger_ratio_drop": None,
            "motion_events_local_path": None,
        })
        continue

    csv_path = DOWNLOAD_DIR / Path(motion_key).name
    if not csv_path.exists():
        print(f"Downloading motion-events csv: {motion_key}")
        s3.download_file(BUCKET, motion_key, str(csv_path))

    try:
        events_df = pd.read_csv(csv_path)
        events_df["event_timestamp"] = pd.to_datetime(
            events_df["timestamp"],
            format="%Y-%m-%d_%H-%M-%S_%f",
            errors="coerce",
        )
        events_df = events_df.dropna(subset=["event_timestamp"]).sort_values("event_timestamp").reset_index(drop=True)

        motion_events_rows = int(len(events_df))
        trigger_count = int(events_df["is_recording_trigger"].fillna(0).astype(int).sum()) if "is_recording_trigger" in events_df.columns else None
        trigger_ratio = (trigger_count / motion_events_rows) if motion_events_rows and trigger_count is not None else None

        if motion_events_rows:
            tail_end = events_df["event_timestamp"].max()
            tail_start = tail_end - pd.Timedelta(seconds=TAIL_WINDOW_SECONDS)
            tail_df = events_df[events_df["event_timestamp"] >= tail_start].copy()
            pre_tail_df = events_df[events_df["event_timestamp"] < tail_start].copy()
        else:
            tail_df = events_df.iloc[0:0].copy()
            pre_tail_df = events_df.iloc[0:0].copy()

        tail_rows = int(len(tail_df))
        pre_tail_rows = int(len(pre_tail_df))
        tail_trigger_count = int(tail_df["is_recording_trigger"].fillna(0).astype(int).sum()) if tail_rows and "is_recording_trigger" in tail_df.columns else 0
        pre_tail_trigger_count = int(pre_tail_df["is_recording_trigger"].fillna(0).astype(int).sum()) if pre_tail_rows and "is_recording_trigger" in pre_tail_df.columns else 0
        tail_trigger_ratio = (tail_trigger_count / tail_rows) if tail_rows else None
        pre_tail_trigger_ratio = (pre_tail_trigger_count / pre_tail_rows) if pre_tail_rows else None
        trigger_ratio_drop = (
            pre_tail_trigger_ratio - tail_trigger_ratio
            if pre_tail_trigger_ratio is not None and tail_trigger_ratio is not None
            else None
        )

        motion_summary_rows.append({
            "key": key,
            "motion_events_rows": motion_events_rows,
            "trigger_count": trigger_count,
            "trigger_ratio": trigger_ratio,
            "pre_tail_rows": pre_tail_rows,
            "pre_tail_trigger_count": pre_tail_trigger_count,
            "pre_tail_trigger_ratio": pre_tail_trigger_ratio,
            "tail_rows": tail_rows,
            "tail_trigger_count": tail_trigger_count,
            "tail_trigger_ratio": tail_trigger_ratio,
            "trigger_ratio_drop": trigger_ratio_drop,
            "motion_events_local_path": str(csv_path),
        })
    except Exception as exc:
        print(f"Could not read {motion_key}: {exc}")
        motion_summary_rows.append({
            "key": key,
            "motion_events_rows": None,
            "trigger_count": None,
            "trigger_ratio": None,
            "pre_tail_rows": None,
            "pre_tail_trigger_count": None,
            "pre_tail_trigger_ratio": None,
            "tail_rows": None,
            "tail_trigger_count": None,
            "tail_trigger_ratio": None,
            "trigger_ratio_drop": None,
            "motion_events_local_path": str(csv_path),
        })

motion_summary_df = pd.DataFrame(motion_summary_rows)
df = df.merge(motion_summary_df, on="key", how="left")

df[[
    "timestamp",
    "key",
    "pre_tail_trigger_ratio",
    "tail_trigger_ratio",
    "trigger_ratio_drop",
    "tail_rows",
]].head()
    


,timestamp,key,pre_tail_trigger_ratio,tail_trigger_ratio,trigger_ratio_drop,tail_rows
0,2026-06-18 21:00:38.579981,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,0.000000,0.0,0.000000,9
1,2026-06-18 21:01:49.748849,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,0.000000,0.0,0.000000,6
2,2026-06-18 21:01:57.793545,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,0.018519,0.0,0.018519,10
3,2026-06-18 21:02:43.500000,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,0.016667,0.0,0.016667,6
4,2026-06-18 21:03:35.890322,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,NaN,0.5,NaN,2


In [ ]:
candidates = df.loc[
    (df["tail_rows"].fillna(0) >= MIN_TAIL_ROWS)
    & (df["tail_trigger_ratio"].fillna(1.0) <= MAX_TAIL_TRIGGER_RATIO),
    [
        "timestamp",
        "key",
        "size_mb",
        "motion_events_key",
        "motion_events_rows",
        "pre_tail_rows",
        "pre_tail_trigger_count",
        "pre_tail_trigger_ratio",
        "tail_rows",
        "tail_trigger_count",
        "tail_trigger_ratio",
        "trigger_ratio_drop",
    ],
].copy()

candidates = candidates.sort_values(
    ["tail_trigger_ratio", "tail_trigger_count", "trigger_ratio_drop"],
    ascending=[True, True, False],
    na_position="last",
).head(MAX_CANDIDATES).reset_index(drop=True)

print("Motionless-tail candidates:", len(candidates))
candidates


Motionless-tail candidates: 274


,timestamp,key,size_mb,motion_events_key,motion_events_rows,pre_tail_rows,pre_tail_trigger_count,pre_tail_trigger_ratio,tail_rows,tail_trigger_count,tail_trigger_ratio,trigger_ratio_drop
0,2026-06-18 22:50:39.496778,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,11.928394,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,7,2,1,0.500000,5,0,0.0,0.500000
1,2026-06-19 00:19:49.000000,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,10.597160,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,8,2,1,0.500000,6,0,0.0,0.500000
2,2026-06-19 00:50:54.174189,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,11.128937,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,7,2,1,0.500000,5,0,0.0,0.500000
3,2026-06-19 00:09:43.948389,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,12.046368,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,10,3,1,0.333333,7,0,0.0,0.333333
4,2026-06-18 21:38:50.550000,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,12.083839,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,11,4,1,0.250000,7,0,0.0,0.250000
...,...,...,...,...,...,...,...,...,...,...,...,...
269,2026-06-18 21:56:43.978174,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,7.093281,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,5,0,0,NaN,5,0,0.0,NaN
270,2026-06-18 22:40:41.245412,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,7.297134,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,9,0,0,NaN,9,0,0.0,NaN
271,2026-06-18 23:40:17.562683,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,5.546539,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,5,0,0,NaN,5,0,0.0,NaN
272,2026-06-19 00:12:59.694132,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,7.697132,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,7,0,0,NaN,7,0,0.0,NaN


In [ ]:
CANDIDATE_VIDEO_DIR = PROCESS_FLAGGED_DIR / "candidate_videos"
CANDIDATE_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

downloaded = []
for row in candidates.itertuples(index=False):
    key = row.key
    output_path = CANDIDATE_VIDEO_DIR / Path(key).name

    if output_path.exists():
        downloaded.append({"key": key, "output_path": str(output_path), "status": "already_exists"})
        continue

    print(f"Downloading {key}")
    s3.download_file(BUCKET, key, str(output_path))
    downloaded.append({"key": key, "output_path": str(output_path), "status": "downloaded"})

downloaded_df = pd.DataFrame(downloaded)
print("Candidate videos saved:", len(downloaded_df))
downloaded_df


Candidate videos saved: 3


,key,output_path,status
0,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,process_flagged\candidate_videos\cortexvpu-01a...,downloaded
1,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,process_flagged\candidate_videos\cortexvpu-01a...,downloaded
2,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,process_flagged\candidate_videos\cortexvpu-01a...,downloaded


In [ ]:
if downloaded_df.empty:
    print("No downloaded candidate videos available for ROI selection.")
else:
    first_video_path = Path(downloaded_df.iloc[0]["output_path"])
    frame = read_preview_frame(first_video_path)

    if frame is None:
        print(f"Could not read preview frame from {first_video_path}")
    else:
        roi_window = "Select ROI on first candidate video"
        selected = cv2.selectROI(roi_window, frame, showCrosshair=True, fromCenter=False)
        cv2.destroyWindow(roi_window)
        x, y, w, h = [int(v) for v in selected]
        ROI = (x, y, w, h) if w > 0 and h > 0 else None
        print("ROI:", ROI)
        print("First video used for ROI:", first_video_path)


In [ ]:
cv_rows = []

for row in downloaded_df.itertuples(index=False):
    key = row.key
    video_path = Path(row.output_path)

    if not video_path.exists():
        cv_rows.append({"key": key, "cv_tail_motion_score": None, "cv_frames_used": 0})
        continue

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cv_rows.append({"key": key, "cv_tail_motion_score": None, "cv_frames_used": 0})
        continue

    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if fps <= 0 or frame_count <= 1:
        cap.release()
        cv_rows.append({"key": key, "cv_tail_motion_score": None, "cv_frames_used": 0})
        continue

    tail_frame_count = max(2, int(round(CV_TAIL_SECONDS * fps)))
    start_frame = max(0, frame_count - tail_frame_count)
    sample_step = max(1, int(round(fps / CV_SAMPLE_FPS)))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    diffs = []
    frames_used = 0
    prev_gray = None
    frame_idx = start_frame

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if (frame_idx - start_frame) % sample_step != 0:
            frame_idx += 1
            continue

        if ROI is not None:
            x, y, w, h = ROI
            frame = frame[y:y + h, x:x + w]
            if frame.size == 0:
                frame_idx += 1
                continue

        height, width = frame.shape[:2]
        resize_width = min(CV_RESIZE_WIDTH, width)
        resize_height = max(1, int(height * (resize_width / width)))
        small = cv2.resize(frame, (resize_width, resize_height))
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        if prev_gray is not None:
            diffs.append(float(np.mean(cv2.absdiff(gray, prev_gray))))
        prev_gray = gray
        frames_used += 1
        frame_idx += 1

    cap.release()
    cv_rows.append({
        "key": key,
        "cv_tail_motion_score": float(np.mean(diffs)) if diffs else None,
        "cv_frames_used": frames_used,
    })

cv_results = candidates.merge(pd.DataFrame(cv_rows), on="key", how="left")
cv_results = cv_results.sort_values(
    ["cv_tail_motion_score", "tail_trigger_ratio", "tail_trigger_count"],
    ascending=[True, True, True],
    na_position="last",
).reset_index(drop=True)

print("ROI used:", ROI)
print("CV-scored candidates:", len(cv_results))
cv_results


CV-scored candidates: 268


,timestamp,key,size_mb,motion_events_key,motion_events_rows,pre_tail_rows,pre_tail_trigger_count,pre_tail_trigger_ratio,tail_rows,tail_trigger_count,tail_trigger_ratio,trigger_ratio_drop,cv_tail_motion_score,cv_frames_used
0,2026-06-18 22:50:39.496778,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,11.928394,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,7,2,1,0.500000,5,0,0.0,0.500000,NaN,NaN
1,2026-06-19 00:19:49.000000,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,10.597160,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,8,2,1,0.500000,6,0,0.0,0.500000,NaN,NaN
2,2026-06-19 00:50:54.174189,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,11.128937,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,7,2,1,0.500000,5,0,0.0,0.500000,NaN,NaN
3,2026-06-19 00:09:43.948389,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,12.046368,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,10,3,1,0.333333,7,0,0.0,0.333333,NaN,NaN
4,2026-06-18 21:38:50.550000,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,12.083839,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,11,4,1,0.250000,7,0,0.0,0.250000,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263,2026-06-19 01:21:03.174615,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,71.530205,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,82,73,0,0.000000,9,0,0.0,0.000000,NaN,NaN
264,2026-06-19 01:22:14.165964,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,37.763119,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,52,46,0,0.000000,6,0,0.0,0.000000,NaN,NaN
265,2026-06-19 01:24:00.125019,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,71.526798,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,129,120,0,0.000000,9,0,0.0,0.000000,NaN,NaN
266,2026-06-19 01:25:11.112602,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,26.249783,cortexvpu-01a-005-41884872/cortexvpu-01a-005-4...,35,26,0,0.000000,9,0,0.0,0.000000,NaN,NaN


In [85]:
import cv2

video_path = r"C:\Users\TimKitchen\OneDrive\OneDrive - PurpleSector\03 Project Specific Documents\24H_INSIGHTS\24H_Insights\process_flagged\candidate_videos\cortexvpu-01a-005-41884872_2026-06-19_00-59-34_800000.ts"
cap = cv2.VideoCapture(video_path)
ok, frame = cap.read()
cap.release()

if not ok:
    raise ValueError("Could not read frame")

roi = cv2.selectROI("Select Conveyor ROI", frame, showCrosshair=True, fromCenter=False)
cv2.destroyAllWindows()

print("ROI =", tuple(int(v) for v in roi))

ROI = (567, 332, 247, 243)
